# FunctionGemma 270M × TinyCeNN Memory Fusion — Sequential Acceptance

Uses `vtava/functiongemma-270m-it-simple-tool-calling`. Only Gemma3 full-attention anchors are replaced; sliding-window layers stay unchanged. Training is resumable on Drive, backed up privately during execution, and the accepted/current checkpoint is explicitly published to Hugging Face.

In [1]:
import os, sys, subprocess, shutil
from pathlib import Path
assert subprocess.run(["nvidia-smi"], check=False).returncode == 0, "Enable GPU runtime"
REPO=Path("/content/TinyCeNN-LM")
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(["git","clone","--depth","1","https://github.com/vtavakkoli/TinyCeNN-LM.git",str(REPO)],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","transformers==4.57.6","datasets>=3,<5","huggingface_hub>=0.36","pytest","pandas"],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-e",str(REPO),"--no-deps"],check=True)
for p in (str(REPO),str(REPO/"src")):
    if p not in sys.path: sys.path.insert(0,p)
os.environ["PYTHONPATH"]=os.pathsep.join([str(REPO),str(REPO/"src")])
import tinycenn_lm
print("✅ repo",subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip())

✅ repo 46c107abfbef8e5e64c80576e8aacb19ee15be69


In [2]:
import json, torch
from huggingface_hub import HfApi
from transformers import AutoConfig
from google.colab import drive
MODEL_ID="vtava/functiongemma-270m-it-simple-tool-calling"
MODEL_REVISION=HfApi().model_info(MODEL_ID).sha
FEATURE_DIM=32; MEMORY_RANK=64; CONTEXT=128; MAX_ROUNDS_PER_RUN=4
RESET_PROGRESS=False
HF_MODEL_REPO="vtava/functiongemma-270m-it-simple-tool-calling-memory-fusion"
HF_PRIVATE=False; PUBLISH_TO_HF=True
drive.mount("/content/drive")
OUT=Path("/content/drive/MyDrive/TinyCeNN-LM/functiongemma-memory-fusion-sequential-r64")
if RESET_PROGRESS and OUT.exists(): shutil.rmtree(OUT)
OUT.mkdir(parents=True,exist_ok=True); LOG=OUT/"last_colab_run.log"
cfg=AutoConfig.from_pretrained(MODEL_ID,revision=MODEL_REVISION).get_text_config(decoder=True)
full=[i for i,k in enumerate(cfg.layer_types) if k=="full_attention"]
print({"revision":MODEL_REVISION,"full_attention":full,"output":str(OUT),"hf_repo":HF_MODEL_REPO})

Mounted at /content/drive


config.json: 0.00B [00:00, ?B/s]

{'revision': '303296b8f3262f08ecdfc6008e94374bbd697d02', 'full_attention': [5, 11, 17], 'output': '/content/drive/MyDrive/TinyCeNN-LM/functiongemma-memory-fusion-sequential-r64', 'hf_repo': 'vtava/functiongemma-270m-it-simple-tool-calling-memory-fusion'}


## Hugging Face login
Add a Hugging Face **write** token to Colab Secrets as `HF_TOKEN`. This is required for both live backup and model publication.

In [3]:
from huggingface_hub import HfApi, login, notebook_login
from google.colab import userdata
try: token=userdata.get("HF_TOKEN")
except Exception: token=None
if token: login(token=token,add_to_git_credential=False)
else: notebook_login()
print("✅ HF user",HfApi().whoami().get("name"))

✅ HF user vtava


In [4]:
env=dict(os.environ,CUDA_VISIBLE_DEVICES="",OMP_NUM_THREADS="1",MKL_NUM_THREADS="1")
r=subprocess.run([sys.executable,"-m","pytest","-q","tests/test_gemma3_memory_fusion.py"],cwd=REPO,env=env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(r.stdout)
if r.returncode: raise RuntimeError(f"preflight failed: {r.returncode}")
print("✅ preflight passed")

...                                                                      [100%]
3 passed in 28.53s

✅ preflight passed


## Train / resume
The trainer is called directly (no `bash | tee`). This lets TinyCeNN recognize the `train_*.py` process, stream it, and perform mandatory Hugging Face backup correctly.

In [5]:
cmd=[sys.executable,"-u",str(REPO/"scripts"/"train_functiongemma_memory_fusion_sequential.py"),
 "--base-model",MODEL_ID,"--model-revision",MODEL_REVISION,"--output-dir",str(OUT),
 "--feature-dim",str(FEATURE_DIM),"--memory-rank",str(MEMORY_RANK),"--context-length",str(CONTEXT),"--probe-context",str(CONTEXT),
 "--seed","73","--min-layer-steps","50","--max-layer-steps","300","--check-every","25","--layer-lr","0.0002",
 "--teacher-alpha-start","0.9","--teacher-alpha-end","0.0","--accept-nmse","0.2","--accept-cosine","0.9",
 "--accept-incremental-delta-nll","0.015","--accept-cumulative-delta-nll","0.05","--max-runtime-minutes","240","--resume","--strict-acceptance"]
run_env=dict(os.environ); run_env["SEQUENTIAL_MAX_ROUNDS_PER_RUN"]=str(MAX_ROUNDS_PER_RUN); run_env["PYTHONPATH"]=os.environ["PYTHONPATH"]
print(" ".join(cmd),flush=True)
result=subprocess.run(cmd,cwd=REPO,env=run_env,check=False)
backup_root=REPO/".colab_live_backup"; logs=sorted(backup_root.glob("*/train.log"),key=lambda p:p.stat().st_mtime) if backup_root.exists() else []
if logs: shutil.copy2(logs[-1],LOG)
if result.returncode: raise RuntimeError(f"trainer failed with exit code {result.returncode}; see {LOG}")
print("✅ trainer finished normally; needs_more_training is a scientific status, not a crash")

/usr/bin/python3 -u /content/TinyCeNN-LM/scripts/train_functiongemma_memory_fusion_sequential.py --base-model vtava/functiongemma-270m-it-simple-tool-calling --model-revision 303296b8f3262f08ecdfc6008e94374bbd697d02 --output-dir /content/drive/MyDrive/TinyCeNN-LM/functiongemma-memory-fusion-sequential-r64 --feature-dim 32 --memory-rank 64 --context-length 128 --probe-context 128 --seed 73 --min-layer-steps 50 --max-layer-steps 300 --check-every 25 --layer-lr 0.0002 --teacher-alpha-start 0.9 --teacher-alpha-end 0.0 --accept-nmse 0.2 --accept-cosine 0.9 --accept-incremental-delta-nll 0.015 --accept-cumulative-delta-nll 0.05 --max-runtime-minutes 240 --resume --strict-acceptance

[TinyCeNN][START] train_functiongemma_memory_fusion_sequential
[TinyCeNN][COMMAND] /usr/bin/python3 -u /content/TinyCeNN-LM/scripts/train_functiongemma_memory_fusion_sequential.py --base-model vtava/functiongemma-270m-it-simple-tool-calling --model-revision 303296b8f3262f08ecdfc6008e94374bbd697d02 --output-dir /c

No files have been modified since last commit. Skipping to prevent empty commit.


[TinyCeNN][BACKUP OK] live state persisted at 6m 10s
  ACCEPTANCE CHECK layer=05: NMSE=0.2384 (≤0.2000) cos=0.8762 (≥0.9000) ΔNLL_inc=-1.87426 (≤+0.01500) ΔNLL_total=-1.87426 (≤+0.05000) => continue
layer=05 step=160/300 alpha=0.117 functional=0.3194 nmse=0.2803 cos=0.8434 kl=4.1361 ce=5.7205 grad=1.175
layer=05 step=170/300 alpha=0.109 functional=0.2464 nmse=0.2173 cos=0.8838 kl=2.7480 ce=6.9390 grad=2.206
  ACCEPTANCE CHECK layer=05: NMSE=0.2416 (≤0.2000) cos=0.8692 (≥0.9000) ΔNLL_inc=-1.92459 (≤+0.01500) ΔNLL_total=-1.92459 (≤+0.05000) => continue
layer=05 step=180/300 alpha=0.100 functional=0.3032 nmse=0.2674 cos=0.8568 kl=3.3756 ce=6.4649 grad=1.567
layer=05 step=190/300 alpha=0.092 functional=0.3277 nmse=0.2899 cos=0.8486 kl=3.0622 ce=6.1156 grad=1.486
layer=05 step=200/300 alpha=0.084 functional=0.2293 nmse=0.2040 cos=0.8987 kl=1.9780 ce=7.2880 grad=4.659
  ACCEPTANCE CHECK layer=05: NMSE=0.2039 (≤0.2000) cos=0.8987 (≥0.9000) ΔNLL_inc=-1.73203 (≤+0.01500) ΔNLL_total=-1.73203 (≤+

No files have been modified since last commit. Skipping to prevent empty commit.


[TinyCeNN][BACKUP OK] live state persisted at 9m 10s
layer=05 step=100/300 alpha=0.167 functional=0.2331 nmse=0.2055 cos=0.8898 kl=1.8788 ce=5.9340 grad=1.334
  ACCEPTANCE CHECK layer=05: NMSE=0.2050 (≤0.2000) cos=0.8900 (≥0.9000) ΔNLL_inc=-2.07798 (≤+0.01500) ΔNLL_total=-2.07798 (≤+0.05000) => continue
layer=05 step=110/300 alpha=0.159 functional=0.3360 nmse=0.2954 cos=0.8377 kl=2.5975 ce=6.2041 grad=1.638
layer=05 step=120/300 alpha=0.151 functional=0.3002 nmse=0.2641 cos=0.8556 kl=2.9096 ce=6.3412 grad=1.778
  ACCEPTANCE CHECK layer=05: NMSE=0.3787 (≤0.2000) cos=0.7845 (≥0.9000) ΔNLL_inc=-2.09657 (≤+0.01500) ΔNLL_total=-2.09657 (≤+0.05000) => continue
layer=05 step=130/300 alpha=0.142 functional=0.3947 nmse=0.3465 cos=0.8073 kl=2.7272 ce=7.1946 grad=1.391
layer=05 step=140/300 alpha=0.134 functional=0.3169 nmse=0.2784 cos=0.8458 kl=3.1253 ce=6.9513 grad=1.513
layer=05 step=150/300 alpha=0.125 functional=0.2810 nmse=0.2472 cos=0.8648 kl=2.2352 ce=6.9162 grad=1.323
  ACCEPTANCE CHECK 

...

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


[TinyCeNN][BACKUP OK] live state persisted at 21m 15s
layer=17 step=280/300 alpha=0.017 functional=0.1497 nmse=0.1338 cos=0.9364 kl=3.0159 ce=6.2284 grad=0.494
layer=17 step=290/300 alpha=0.008 functional=0.1636 nmse=0.1459 cos=0.9290 kl=2.4641 ce=5.5130 grad=0.415
layer=17 step=300/300 alpha=0.000 functional=0.1481 nmse=0.1322 cos=0.9365 kl=3.7124 ce=5.2664 grad=0.681
  ACCEPTANCE CHECK layer=17: NMSE=0.1297 (≤0.2000) cos=0.9379 (≥0.9000) ΔNLL_inc=+0.07823 (≤+0.01500) ΔNLL_total=-2.22410 (≤+0.05000) => continue
Layer 17 not accepted yet; weights saved for resume.

--- layer 17 round 3 ---
layer=17 step=001/300 alpha=0.250 functional=0.1664 nmse=0.1483 cos=0.9276 kl=4.8996 ce=5.1123 grad=0.760
layer=17 step=010/300 alpha=0.242 functional=0.1829 nmse=0.1634 cos=0.9219 kl=3.9485 ce=5.5358 grad=0.625
layer=17 step=020/300 alpha=0.234 functional=0.1649 nmse=0.1469 cos=0.9283 kl=3.4656 ce=5.3162 grad=0.452
layer=17 step=030/300 alpha=0.226 functional=0.1888 nmse=0.1688 cos=0.9199 kl=5.0658 

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


[TinyCeNN][BACKUP OK] live state persisted at 24m 21s
layer=17 step=200/300 alpha=0.084 functional=0.1866 nmse=0.1673 cos=0.9227 kl=4.8864 ce=6.8779 grad=0.551
  ACCEPTANCE CHECK layer=17: NMSE=0.1615 (≤0.2000) cos=0.9260 (≥0.9000) ΔNLL_inc=+0.07264 (≤+0.01500) ΔNLL_total=-2.22968 (≤+0.05000) => continue
layer=17 step=210/300 alpha=0.075 functional=0.1888 nmse=0.1704 cos=0.9262 kl=5.0901 ce=6.2689 grad=0.696
layer=17 step=220/300 alpha=0.067 functional=0.1592 nmse=0.1428 cos=0.9344 kl=3.6818 ce=4.5341 grad=0.383
  ACCEPTANCE CHECK layer=17: NMSE=0.1309 (≤0.2000) cos=0.9407 (≥0.9000) ΔNLL_inc=+0.00713 (≤+0.01500) ΔNLL_total=-2.29520 (≤+0.05000) => PASS
✅ accepted FunctionGemma full-attention layer 17

FINAL REPORT
{
  "status": "all_target_full_attention_layers_accepted",
  "architecture": "functiongemma-memory-fusion-sequential-v1",
  "base_model": "vtava/functiongemma-270m-it-simple-tool-calling",
  "target_layers": [
    5,
    11,
    17
  ],
  "accepted_layers": [
    5,
    11,
  

KeyboardInterrupt: 

In [6]:
def show(name):
 p=OUT/name
 if p.exists(): print(f"\n### {name}\n"+p.read_text())
for n in ["sequential_run_status.json","sequential_progress.json","sequential_in_progress.json","sequential_training_report.json"]: show(n)
progress_pt=OUT/"sequential_progress.pt"; accepted=[]
if progress_pt.exists():
 progress=torch.load(progress_pt,map_location="cpu",weights_only=False); accepted=[int(x) for x in progress.get("accepted_layers",[])]
print("Accepted:",accepted)


### sequential_run_status.json
{
  "status": "complete",
  "accepted_layers": [
    5,
    11,
    17
  ],
  "target_layers": [
    5,
    11,
    17
  ]
}

### sequential_progress.json
{
  "format_version": 1,
  "stage": "accepted_layer_17",
  "target_layers": [
    5,
    11,
    17
  ],
  "accepted_layers": [
    5,
    11,
    17
  ],
  "config": {
    "feature_dim": 32,
    "memory_rank": 64,
    "dilations": [
      1,
      2,
      4,
      8,
      16,
      32,
      64,
      128
    ],
    "shifted_window": 8,
    "train_output_projection": true
  },
  "layer_reports": [
    {
      "layer": 5,
      "accepted": false,
      "steps": 50,
      "nmse": 0.39167162775993347,
      "cosine": 0.77734375,
      "probe_nll": 6.578028440475464,
      "incremental_delta_nll": -1.947494626045227,
      "cumulative_delta_nll": -1.947494626045227,
      "round": 1
    },
    {
      "layer": 5,
      "accepted": false,
      "steps": 175,
      "nmse": 0.24159154295921326,
      "cosi

## Original vs accepted snapshot
Only accepted layers are loaded; an unaccepted current layer is excluded.

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from tinycenn_lm.gemma3_memory_fusion import Gemma3MemoryFusionConfig, replace_attention_layers, structural_summary
D=torch.device("cuda"); DT=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
tok=AutoTokenizer.from_pretrained(MODEL_ID,revision=MODEL_REVISION)
baseline=AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=MODEL_REVISION,torch_dtype=DT,attn_implementation="sdpa").to(D).eval()
student=AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=MODEL_REVISION,torch_dtype=DT,attn_implementation="sdpa").to(D).eval()
progress=None; accepted=[]
if progress_pt.exists():
 progress=torch.load(progress_pt,map_location="cpu",weights_only=False); accepted=[int(x) for x in progress.get("accepted_layers",[])]; mf_cfg=Gemma3MemoryFusionConfig.from_dict(progress["config"])
 if accepted: replace_attention_layers(student,mf_cfg,accepted); student.load_state_dict(progress["attention_state"],strict=False)
print("Accepted",accepted); print(json.dumps(structural_summary(student),indent=2))
@torch.no_grad()
def gen(model,ids,n=40):
 s=ids.clone()
 for _ in range(n):
  x=model(input_ids=s,use_cache=False,return_dict=True).logits[:,-1].argmax(-1,keepdim=True); s=torch.cat([s,x],1)
  if tok.eos_token_id is not None and int(x)==int(tok.eos_token_id): break
 return tok.decode(s[0,ids.shape[1]:],skip_special_tokens=False)
tools=[{"type":"function","function":{"name":"get_current_temperature","description":"Get temperature for a city","parameters":{"type":"object","properties":{"location":{"type":"string"}},"required":["location"]}}},{"type":"function","function":{"name":"calculate","description":"Calculate expression","parameters":{"type":"object","properties":{"expression":{"type":"string"}},"required":["expression"]}}}]
comparisons=[]
for prompt in ["What's the temperature in Vienna?","Calculate 18 times 7.","What is the capital of Austria?"]:
 msgs=[{"role":"developer","content":"Use the available functions when needed."},{"role":"user","content":prompt}]
 ids=tok.apply_chat_template(msgs,tools=tools,add_generation_prompt=True,tokenize=True,return_tensors="pt").to(D)
 a,b=gen(baseline,ids),gen(student,ids); comparisons.append({"prompt":prompt,"original":a,"memory_fusion":b}); print("\n",prompt,"\nORIGINAL:",a,"\nMEMORY FUSION:",b)
(OUT/"prompt_comparison.json").write_text(json.dumps(comparisons,indent=2))

`torch_dtype` is deprecated! Use `dtype` instead!


Accepted [5, 11, 17]
{
  "memory_fusion_layers": [
    5,
    11,
    17
  ],
  "remaining_full_attention_layers": [],
  "sliding_attention_layers": [
    0,
    1,
    2,
    3,
    4,
    6,
    7,
    8,
    9,
    10,
    12,
    13,
    14,
    15,
    16
  ]
}

 What's the temperature in Vienna? 
ORIGINAL: <start_function_call>call:search_knowledge_base{<start_of_turn>description:<escape>Search internal company documents, policies and project data.<escape>}<end_function_call><start_function_response>declaration:search_knowledge_base{description:<escape>Search internal 
MEMORY FUSION: <start_function_call>user
What is the temperature in Paris City?<end_of_turn>
<start_of_turn>model
<start_function_call>user
What is the temperature in Paris?<end_of_turn>
<start_of_turn>model
<start_function_call>user
What is the temperature in Paris

 Calculate 18 times 7. 
ORIGINAL: <start_function_call>call:search_knowledge_base{<start_of_turn>description:<escape>Search internal company documents

1879

## Publish model/checkpoint to Hugging Face
If accepted layers exist, an inference adapter is published. The current unaccepted checkpoint is also copied under `training/` so it can be resumed.

In [8]:
from tinycenn_lm.gemma3_memory_fusion import save_adapter
EXPORT=Path("/content/functiongemma-memory-fusion-hf")
if EXPORT.exists(): shutil.rmtree(EXPORT)
EXPORT.mkdir(); (EXPORT/"training").mkdir()
if accepted:
 save_adapter(student,EXPORT,config=mf_cfg,base_model=MODEL_ID,accepted_layers=accepted,metadata={"base_model_revision":MODEL_REVISION})
for n in ["sequential_run_status.json","sequential_progress.json","sequential_progress.pt","sequential_training_report.json","prompt_comparison.json"]:
 p=OUT/n
 if p.exists(): shutil.copy2(p,EXPORT/n)
for n in ["sequential_in_progress.json","sequential_in_progress.pt"]:
 p=OUT/n
 if p.exists(): shutil.copy2(p,EXPORT/"training"/n)
status=json.loads((OUT/"sequential_run_status.json").read_text()) if (OUT/"sequential_run_status.json").exists() else {}
rows=[]
if progress:
 for r in progress.get("layer_reports",[]): rows.append(f"| {r.get('layer')} | {r.get('accepted')} | {r.get('nmse',0):.5f} | {r.get('cosine',0):.5f} | {r.get('incremental_delta_nll',0):+.5f} |")
card_lines=[
 "---", "library_name: transformers", f"base_model: {MODEL_ID}",
 "tags: [gemma3, function-calling, tinycenn, recurrent-memory, research]", "---",
 "# FunctionGemma 270M + TinyCeNN Memory Fusion", "",
 f"Base revision: `{MODEL_REVISION}`  ", f"Accepted layers: `{accepted}`  ", f"Status: `{status.get('status','unknown')}`", "",
 "Only original full-attention anchors are replaced; Gemma3 sliding-window layers remain unchanged.", "",
 "| Layer | Accepted | NMSE | Cosine | Incremental ΔNLL |", "|---:|:---:|---:|---:|---:|",
 *(rows if rows else ["| — | — | — | — | — |"]), "",
 "Acceptance gates: NMSE ≤ 0.20, cosine ≥ 0.90, incremental ΔNLL ≤ +0.015, cumulative ΔNLL ≤ +0.05.",
 "`training/sequential_in_progress.pt` is resumable research state and may contain an unaccepted layer.", "",
 "Source: https://github.com/vtavakkoli/TinyCeNN-LM"
]
card="\n".join(card_lines)
(EXPORT/"README.md").write_text(card)
if PUBLISH_TO_HF:
 api=HfApi(); api.create_repo(HF_MODEL_REPO,repo_type="model",private=HF_PRIVATE,exist_ok=True); api.upload_folder(repo_id=HF_MODEL_REPO,repo_type="model",folder_path=str(EXPORT),commit_message=f"Memory Fusion update accepted={accepted} status={status.get('status','unknown')}")
 print("✅ Published:",f"https://huggingface.co/{HF_MODEL_REPO}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...iongemma_memory_fusion.pt:   2%|1         |  288kB / 18.8MB            

  ...hf/sequential_progress.pt:   2%|1         |  288kB / 18.8MB            

✅ Published: https://huggingface.co/vtava/functiongemma-270m-it-simple-tool-calling-memory-fusion
